# TEP Anomaly Classification Benchmark — Fixed
### Tennessee Eastman Process — Deep & Shallow Learning mit DR-Vergleich

**Dimensionsreduktion:** `NONE` | `PCA` | `LDA`  
**Modelle:** LSTM-FCN · Deep CNN · TCN · RNN · LSTM · WaveNet · XGBoost · Random Forest · SVM  
**Hyperparameter-Tuning:** GridSearchCV für XGBoost, Random Forest und SVM

---
### Fixes gegenüber letzter Version
| # | Problem | Fix |
|---|---------|-----|
| 1 | **TCN RF=90 << 500** — Modell lernte nichts | `dilations=[1,2,4,8,16,32,64,128]` → RF=1530 |
| 2 | **EarlyStopping zu aggressiv** für RNN/LSTM | `patience=12` + `EPOCHS=100` für langsame Modelle |
| 3 | **SVM/XGBoost|LDA Label-Bug** | Globaler `LabelEncoder`, einmalig auf [1..20] gefittet |
| 4 | **val_loss statt val_accuracy** | Glatterer Monitor, weniger Fehlalarme |

---
**Benötigte CSV-Dateien (gleicher Ordner):**
- `TEP_Faulty_Training.csv`
- `TEP_Faulty_Testing.csv`

## 1 · Imports & Konfiguration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    LSTM, Activation, Add, BatchNormalization, Concatenate, Conv1D,
    Dense, Dropout, GlobalAveragePooling1D, Input, Multiply, SimpleRNN,
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam

try:
    from tcn import TCN
    TCN_AVAILABLE = True
    print("✓ keras-tcn verfügbar")
except ImportError:
    TCN_AVAILABLE = False
    print("⚠ keras-tcn nicht installiert → pip install keras-tcn")

print(f"TensorFlow {tf.__version__} | Pandas {pd.__version__} | NumPy {np.__version__}")

✓ keras-tcn verfügbar
TensorFlow 2.21.0 | Pandas 3.0.2 | NumPy 2.4.4


## 2 · Konstanten

In [2]:
META_COLS     = ['faultNumber', 'simulationRun', 'sample']
SEQ_LEN_TRAIN = 500
SEQ_LEN_TEST  = 960
BATCH_SIZE    = 32

# ── FIX 3: Globaler LabelEncoder — einmalig auf faultNumber 1..20 gefittet ──
# Alle Branches (Deep & Shallow, alle DR) verwenden diesen EINEN Encoder.
# Das verhindert den Label-Mapping-Bug bei SVM/XGBoost mit LDA.
GLOBAL_LE = LabelEncoder()
GLOBAL_LE.fit(np.arange(1, 21))   # Klassen: 1, 2, ..., 20
N_CLASSES = len(GLOBAL_LE.classes_)
print(f"Label-Klassen: {GLOBAL_LE.classes_}  →  {N_CLASSES} Klassen")

# ── FIX 2: Modell-spezifische Callbacks & Epochen ───────────────────────────
# FAST: LSTM-FCN, Deep CNN, WaveNet  → konvergieren schnell, früher Stopp ok
# SLOW: RNN, LSTM, TCN               → brauchen viel mehr Epochen
def make_callbacks_fast():
    return [
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.333,
                          patience=3, min_lr=1e-6, mode='min'),
    ]

def make_callbacks_slow():
    return [
        EarlyStopping(monitor='val_loss', patience=12,
                      restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.333,
                          patience=4, min_lr=1e-6, mode='min'),
    ]

EPOCHS_FAST = 50
EPOCHS_SLOW = 100

# Jedes Modell bekommt sein Callback-Profil
CALLBACK_MAP = {
    'LSTM-FCN': (make_callbacks_fast, EPOCHS_FAST),
    'Deep CNN': (make_callbacks_fast, EPOCHS_FAST),
    'WaveNet' : (make_callbacks_fast, EPOCHS_FAST),
    'RNN'     : (make_callbacks_slow, EPOCHS_SLOW),
    'LSTM'    : (make_callbacks_slow, EPOCHS_SLOW),
    'TCN'     : (make_callbacks_slow, EPOCHS_SLOW),
}

# ── GridSearch-Grids ─────────────────────────────────────────────────────────
GRID_XGB = {
    'n_estimators' : [200, 300, 500],
    'max_depth'    : [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
}
GRID_RF = {
    'n_estimators'    : [200, 300, 500],
    'max_depth'       : [None, 10, 20],
    'min_samples_split': [2, 5],
}
GRID_SVM = {
    'C'     : [1, 10, 100],
    'gamma' : ['scale', 'auto'],
    'kernel': ['rbf'],
}

Label-Klassen: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]  →  20 Klassen


## 3 · Daten laden

Nur die **Faulty-Dateien** werden geladen (faultNumber 1–20).  
Der Benchmark fokussiert auf Anomaly Classification — FaultFree-Daten (faultNumber 0) werden nicht benötigt.

In [3]:
def load_tep_data():
    """Lädt ausschließlich die Faulty-Dateien (faultNumber 1–20)."""
    def _load(path):
        header = pd.read_csv(path, nrows=0).columns.tolist()
        dtypes = {c: np.float32 for c in header if c not in META_COLS}
        return pd.read_csv(path, dtype=dtypes)

    df_train = _load("TEP_Faulty_Training.csv")
    df_test  = _load("TEP_Faulty_Testing.csv")

    print(f"Train : {df_train.shape} | Klassen: {sorted(df_train['faultNumber'].unique())}")
    print(f"Test  : {df_test.shape}  | Klassen: {sorted(df_test['faultNumber'].unique())}")
    return df_train, df_test

df_train, df_test = load_tep_data()

Train : (5000000, 55) | Klassen: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]
Test  : (9600000, 55)  | Klassen: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]


## 4 · Preprocessing-Hilfsfunktionen

In [4]:
def _split_xy(df):
    feat = [c for c in df.columns if c not in META_COLS]
    return df[feat], df['faultNumber'], df['simulationRun']

def create_sequences(df_2d, labels_1d, runs_1d, seq_len):
    """Nicht-überlappende Sequenzen, Gruppierung nach simulationRun."""
    df = pd.DataFrame(df_2d)
    df['_run']   = runs_1d.values
    df['_label'] = labels_1d.values
    feat_cols = [c for c in df.columns if c not in ('_run', '_label')]
    seqs, labels = [], []
    for _, g in df.groupby('_run'):
        d, l = g[feat_cols].values, g['_label'].values
        for i in range(0, len(d) - seq_len + 1, seq_len):
            seqs.append(d[i : i + seq_len])
            labels.append(l[i + seq_len - 1])
    return np.array(seqs, dtype=np.float32), np.array(labels)

def create_aggregated(df_2d, labels_1d, runs_1d, window):
    """Mittelt Zeitfenster der Länge window → flache Feature-Vektoren."""
    df = pd.DataFrame(df_2d)
    df['_run']   = runs_1d.values
    df['_label'] = labels_1d.values
    feat_cols = [c for c in df.columns if c not in ('_run', '_label')]
    agg_X, agg_y = [], []
    for _, g in df.groupby('_run'):
        d, l = g[feat_cols].values, g['_label'].values
        for i in range(0, len(d) - window + 1, window):
            agg_X.append(d[i : i + window].mean(axis=0))
            agg_y.append(l[i + window - 1])
    return np.array(agg_X, dtype=np.float32), np.array(agg_y)

## 5 · Preprocessing-Pipelines

In [5]:
def preprocess_deep(df_train, df_test, dr='lda'):
    """
    Skalierung → optionale DR → Sequenzbildung.
    Gibt kodierte Labels mit dem GLOBALEN LabelEncoder zurück.

    Hinweis zur LDA-Dimension:
      Deep-Branch: LDA auf (n_samples × 52) → max 19 Komponenten
      Shallow-Branch: LDA auf (n_windows × 52) → identische Dimension
      Beide Branches teilen denselben GLOBAL_LE, daher kein Label-Mismatch.
    """
    X_tr, y_tr, runs_tr = _split_xy(df_train)
    X_te, y_te, runs_te = _split_xy(df_test)

    scaler   = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr).astype(np.float32)
    X_te_sc  = scaler.transform(X_te).astype(np.float32)

    n_cls = N_CLASSES
    if dr == 'lda':
        red     = LDA()
        X_tr_dr = red.fit_transform(X_tr_sc, y_tr).astype(np.float32)
        X_te_dr = red.transform(X_te_sc).astype(np.float32)
    elif dr == 'pca':
        red     = PCA(n_components=n_cls - 1)
        X_tr_dr = red.fit_transform(X_tr_sc).astype(np.float32)
        X_te_dr = red.transform(X_te_sc).astype(np.float32)
    else:
        X_tr_dr, X_te_dr = X_tr_sc, X_te_sc

    X_tr_seq, y_tr_raw = create_sequences(X_tr_dr, y_tr, runs_tr, SEQ_LEN_TRAIN)
    X_te_seq, y_te_raw = create_sequences(X_te_dr, y_te, runs_te, SEQ_LEN_TEST)

    # FIX 3: Globaler Encoder — kein erneutes fit_transform
    y_tr_seq = GLOBAL_LE.transform(y_tr_raw)
    y_te_seq = GLOBAL_LE.transform(y_te_raw)

    print(f"  Deep | DR={dr.upper():<4} | Train {X_tr_seq.shape} | Test {X_te_seq.shape}")
    return X_tr_seq, y_tr_seq, X_te_seq, y_te_seq


def preprocess_shallow(df_train, df_test, dr='lda'):
    """Aggregation (Mittelwert pro Fenster) → optionale DR → flache Vektoren."""
    X_tr, y_tr, runs_tr = _split_xy(df_train)
    X_te, y_te, runs_te = _split_xy(df_test)

    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr).astype(np.float32)
    X_te_sc = scaler.transform(X_te).astype(np.float32)

    X_tr_agg, y_tr_raw = create_aggregated(X_tr_sc, y_tr, runs_tr, SEQ_LEN_TRAIN)
    X_te_agg, y_te_raw = create_aggregated(X_te_sc, y_te, runs_te, SEQ_LEN_TEST)

    n_cls = N_CLASSES
    if dr == 'lda':
        red      = LDA()
        X_tr_agg = red.fit_transform(X_tr_agg, y_tr_raw).astype(np.float32)
        X_te_agg = red.transform(X_te_agg).astype(np.float32)
    elif dr == 'pca':
        red      = PCA(n_components=n_cls - 1)
        X_tr_agg = red.fit_transform(X_tr_agg).astype(np.float32)
        X_te_agg = red.transform(X_te_agg).astype(np.float32)

    # FIX 3: Globaler Encoder
    y_tr_agg_enc = GLOBAL_LE.transform(y_tr_raw)
    y_te_agg_enc = GLOBAL_LE.transform(y_te_raw)

    print(f"  Shallow | DR={dr.upper():<4} | Train {X_tr_agg.shape} | Test {X_te_agg.shape}")
    return X_tr_agg, y_tr_agg_enc, X_te_agg, y_te_agg_enc

## 6 · Modell-Definitionen

In [6]:
def build_lstm_fcn(n_features, n_classes):
    inputs = Input(shape=(None, n_features))
    x1 = LSTM(128)(inputs)
    x1 = Dropout(0.8)(x1)
    x2 = Conv1D(128, 8, padding='same')(inputs)
    x2 = BatchNormalization()(x2); x2 = Activation('relu')(x2)
    x2 = Conv1D(256, 5, padding='same')(x2)
    x2 = BatchNormalization()(x2); x2 = Activation('relu')(x2)
    x2 = Conv1D(128, 3, padding='same')(x2)
    x2 = BatchNormalization()(x2); x2 = Activation('relu')(x2)
    x2 = GlobalAveragePooling1D()(x2)
    out = Dense(n_classes, activation='softmax')(Concatenate()([x1, x2]))
    return Model(inputs, out, name='LSTM_FCN')


def build_deep_cnn(n_features, n_classes):
    m = Sequential(name='Deep_CNN')
    m.add(Input(shape=(None, n_features)))
    for f in [64, 128, 256, 128]:
        m.add(Conv1D(f, 3, padding='same'))
        m.add(BatchNormalization())
        m.add(Activation('relu'))
    m.add(GlobalAveragePooling1D())
    m.add(Dropout(0.5))
    m.add(Dense(128, activation='relu'))
    m.add(Dense(n_classes, activation='softmax'))
    return m


def build_tcn(n_features, n_classes):
    """
    FIX 1: Dilationen auf [1,2,4,8,16,32,64,128] erweitert.
    Receptive Field = 2 × kernel_size × sum(dilations)
                    = 2 × 3 × 255 = 1530  >>  SEQ_LEN_TRAIN=500 ✓
    Vorher: [1,2,4,8,16] → RF=90, zu kurz für 500-Schritt-Sequenzen.
    """
    if not TCN_AVAILABLE:
        raise ImportError("pip install keras-tcn")
    inputs = Input(shape=(None, n_features))
    x = TCN(
        nb_filters=64,
        kernel_size=3,
        nb_stacks=2,
        dilations=[1, 2, 4, 8, 16, 32, 64, 128],   # FIX: war [1,2,4,8,16]
        padding='causal',
        use_skip_connections=True,
        dropout_rate=0.2,
        return_sequences=False,
    )(inputs)
    x = Dense(64, activation='relu')(x)
    out = Dense(n_classes, activation='softmax')(x)
    return Model(inputs, out, name='TCN')


def build_rnn(n_features, n_classes):
    m = Sequential(name='RNN')
    m.add(Input(shape=(None, n_features)))
    m.add(SimpleRNN(128, return_sequences=True))
    m.add(SimpleRNN(64))
    m.add(Dropout(0.3))
    m.add(Dense(n_classes, activation='softmax'))
    return m


def build_lstm(n_features, n_classes):
    m = Sequential(name='LSTM')
    m.add(Input(shape=(None, n_features)))
    m.add(LSTM(128, return_sequences=True))
    m.add(LSTM(64))
    m.add(Dropout(0.3))
    m.add(Dense(n_classes, activation='softmax'))
    return m


def build_wavenet(n_features, n_classes):
    FILTERS, DILATIONS = 32, [1, 2, 4, 8, 16, 32]
    inputs = Input(shape=(None, n_features))
    x = Conv1D(FILTERS, 1, padding='causal')(inputs)
    skips = []
    for d in DILATIONS:
        xt = Conv1D(FILTERS, 2, dilation_rate=d, padding='causal', activation='tanh')(x)
        xs = Conv1D(FILTERS, 2, dilation_rate=d, padding='causal', activation='sigmoid')(x)
        xr = Conv1D(FILTERS, 1)(Multiply()([xt, xs]))
        skips.append(xr)
        x = Add()([x, xr])
    x = Activation('relu')(Add()(skips))
    x = GlobalAveragePooling1D()(Conv1D(FILTERS, 1, activation='relu')(x))
    out = Dense(n_classes, activation='softmax')(Dropout(0.3)(x))
    return Model(inputs, out, name='WaveNet')

## 7 · Training & Evaluierung

In [7]:
def train_eval_deep(model, X_tr, y_tr, X_te, y_te, name):
    """
    FIX 2: Modellname → CALLBACK_MAP → passendes Callback-Profil & Epochen-Limit.
    FIX 4: monitor='val_loss' (glatter, weniger Fehlalarme als val_accuracy).
    """
    model_key = name.split(' | ')[0]          # z.B. 'LSTM-FCN' aus 'LSTM-FCN | LDA'
    cb_factory, max_epochs = CALLBACK_MAP.get(model_key, (make_callbacks_fast, EPOCHS_FAST))
    callbacks = cb_factory()                  # neue Callback-Instanzen pro Lauf

    model.compile(
        optimizer=Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    print(f"\n{'='*62}\n  {name}")
    print(f"  Fit {X_tr.shape} | Val (20 %) | Test {X_te.shape}")
    print(f"  Callbacks: {'SLOW' if max_epochs == EPOCHS_SLOW else 'FAST'} "
          f"(patience={'12' if max_epochs == EPOCHS_SLOW else '5'}, max_epochs={max_epochs})")
    print(f"{'='*62}")

    model.fit(
        X_tr, y_tr,
        epochs=max_epochs,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=1,
    )

    y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
    f1 = f1_score(y_te, y_pred, average='macro')
    print(f"\n[{name}] Makro F1 = {f1:.4f}")
    print(classification_report(
        y_te, y_pred,
        target_names=[f"Fault {c}" for c in GLOBAL_LE.classes_],
        zero_division=0,
    ))
    return f1


def train_eval_shallow(model, X_tr, y_tr, X_te, y_te, name, param_grid=None):
    import itertools
    print(f"\n{'='*62}\n  {name}\n  Train {X_tr.shape} | Test {X_te.shape}")
    if param_grid:
        n_combos = len(list(itertools.product(*param_grid.values())))
        print(f"  GridSearch läuft... ({n_combos} Kombinationen × 3 Folds)")
        gs = GridSearchCV(model, param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
        gs.fit(X_tr, y_tr)
        best = gs.best_estimator_
        print(f"  Beste Parameter : {gs.best_params_}")
        print(f"  Bester CV F1    : {gs.best_score_:.4f}")
    else:
        best = model
        best.fit(X_tr, y_tr)
    print('='*62)

    y_pred = best.predict(X_te)
    f1 = f1_score(y_te, y_pred, average='macro')
    print(f"[{name}] Makro F1 = {f1:.4f}")
    print(classification_report(
        y_te, y_pred,
        target_names=[f"Fault {c}" for c in GLOBAL_LE.classes_],
        zero_division=0,
    ))
    return f1

## 8 · Haupt-Benchmark-Loop

In [8]:
results = {}

for dr in ['none', 'pca', 'lda']:
    print(f"\n{'#'*62}\n  Dimensionsreduktion: {dr.upper()}\n{'#'*62}")

    # ── Deep-Modelle ──────────────────────────────────────────────────────────
    X_tr_seq, y_tr_seq, X_te_seq, y_te_seq = preprocess_deep(df_train, df_test, dr=dr)
    n_cls  = N_CLASSES
    n_feat = X_tr_seq.shape[2]

    deep_models = {
        'LSTM-FCN': build_lstm_fcn(n_feat, n_cls),
        'Deep CNN': build_deep_cnn(n_feat, n_cls),
        'RNN'     : build_rnn(n_feat, n_cls),
        'LSTM'    : build_lstm(n_feat, n_cls),
        'WaveNet' : build_wavenet(n_feat, n_cls),
    }
    if TCN_AVAILABLE:
        deep_models['TCN'] = build_tcn(n_feat, n_cls)

    for name, model in deep_models.items():
        key = f"{name} | {dr.upper()}"
        results[key] = train_eval_deep(
            model, X_tr_seq, y_tr_seq, X_te_seq, y_te_seq, key
        )

    # ── Shallow-Modelle mit GridSearch ────────────────────────────────────────
    X_tr_agg, y_tr_agg, X_te_agg, y_te_agg = preprocess_shallow(df_train, df_test, dr=dr)

    shallow_configs = [
        ('XGBoost',       XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=-1), GRID_XGB),
        ('Random Forest', RandomForestClassifier(random_state=42, n_jobs=-1),                GRID_RF),
        ('SVM',           SVC(decision_function_shape='ovr'),                                GRID_SVM),
    ]
    for name, model, grid in shallow_configs:
        key = f"{name} | {dr.upper()}"
        results[key] = train_eval_shallow(
            model, X_tr_agg, y_tr_agg, X_te_agg, y_te_agg, key, param_grid=grid
        )


##############################################################
  Dimensionsreduktion: NONE
##############################################################
  Deep | DR=NONE | Train (10000, 500, 52) | Test (10000, 960, 52)


  LSTM-FCN | NONE
  Fit (10000, 500, 52) | Val (20 %) | Test (10000, 960, 52)
  Callbacks: FAST (patience=5, max_epochs=50)
Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 77s 297ms/step - accuracy: 0.8055 - loss: 0.5900 - val_accuracy: 0.9015 - val_loss: 0.2274 - learning_rate: 0.0010
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 79s 315ms/step - accuracy: 0.9110 - loss: 0.1808 - val_accuracy: 0.9385 - val_loss: 0.1616 - learning_rate: 0.0010
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 86s 345ms/step - accuracy: 0.9249 - loss: 0.1559 - val_accuracy: 0.9000 - val_loss: 0.2518 - learning_rate: 0.0010
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 93s 371ms/step - accuracy: 0.9471 - loss: 0.1140 - val_accuracy: 0.9480 - val_loss: 0.1033 - learning_rate: 0.0010
Epoch 5/50
250/250 ━━━━━━━━━━━━━━

## 9 · Ergebnistabelle

In [9]:
model_names = ['LSTM-FCN', 'Deep CNN', 'TCN', 'RNN', 'LSTM',
               'WaveNet', 'XGBoost', 'Random Forest', 'SVM']

paper = {
    'LSTM-FCN':     {'NONE': 0.90, 'PCA': 0.89, 'LDA': 0.98},
    'Deep CNN':     {'NONE': 0.96, 'PCA': 0.86, 'LDA': 0.91},
    'TCN':          {'NONE': 0.90, 'PCA': 0.89, 'LDA': 0.93},
    'RNN':          {'NONE': 0.90, 'PCA': 0.78, 'LDA': 0.91},
    'LSTM':         {'NONE': 0.84, 'PCA': 0.71, 'LDA': 0.88},
    'XGBoost':      {'NONE': 0.72, 'PCA': 0.66, 'LDA': 0.74},
    'Random Forest':{'NONE': 0.70, 'PCA': 0.62, 'LDA': 0.77},
    'SVM':          {'NONE': 0.62, 'PCA': 0.60, 'LDA': 0.73},
    'WaveNet':      {'NONE': 0.62, 'PCA': 0.44, 'LDA': 0.66},
}

print("\n" + "="*80)
print(f"  {'Modell':<16} {'No DR':>8} {'PCA':>8} {'LDA':>8} {'AVG':>8}  "
      f"  {'Δ No DR':>8} {'Δ PCA':>8} {'Δ LDA':>8}")
print("="*80)

for m in model_names:
    vals = [results.get(f"{m} | {dr}", float('nan')) for dr in ['NONE', 'PCA', 'LDA']]
    p    = [paper.get(m, {}).get(dr, float('nan'))   for dr in ['NONE', 'PCA', 'LDA']]
    deltas = [v - q if not (np.isnan(v) or np.isnan(q)) else float('nan')
              for v, q in zip(vals, p)]
    row  = f"  {m:<16}" + "".join(f" {v:>8.4f}" if not np.isnan(v) else f" {'—':>8}" for v in vals)
    row += f" {np.nanmean(vals):>8.4f}"
    row += "  " + "".join(f" {d:>+8.2f}" if not np.isnan(d) else f" {'—':>8}" for d in deltas)
    print(row)

print("="*80)
print("\nPaper-Referenzwerte (AVG-Spalte):")
for m in model_names:
    p = paper.get(m, {})
    avg = np.mean(list(p.values()))
    print(f"  {m:<16}  NONE={p.get('NONE','—'):.2f}  PCA={p.get('PCA','—'):.2f}  "
          f"LDA={p.get('LDA','—'):.2f}  AVG={avg:.2f}")


  Modell              No DR      PCA      LDA      AVG     Δ No DR    Δ PCA    Δ LDA
  LSTM-FCN           0.9925   0.9452   0.9408   0.9595      +0.09    +0.06    -0.04
  Deep CNN           0.8922   0.9540   0.9626   0.9363      -0.07    +0.09    +0.05
  TCN                0.7890   0.6967   0.8877   0.7911      -0.11    -0.19    -0.04
  RNN                0.8088   0.7646   0.8006   0.7913      -0.09    -0.02    -0.11
  LSTM               0.8556   0.8729   0.8263   0.8516      +0.02    +0.16    -0.05
  WaveNet            0.9996   0.9795   0.9975   0.9922      +0.38    +0.54    +0.34
  XGBoost            0.7596   0.7049   0.7214   0.7286      +0.04    +0.04    -0.02
  Random Forest      0.8428   0.7890   0.7656   0.7991      +0.14    +0.17    -0.00
  SVM                0.7749   0.7413   0.5743   0.6968      +0.15    +0.14    -0.16

Paper-Referenzwerte (AVG-Spalte):
  LSTM-FCN          NONE=0.90  PCA=0.89  LDA=0.98  AVG=0.92
  Deep CNN          NONE=0.96  PCA=0.86  LDA=0.91  AVG=0.91
  T